In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [4]:
df = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding = "latin-1", header = None)
df

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


In [5]:
df = df[[0,5]]
df.columns = ["polarity", "text"]
print(df.head())

   polarity                                               text
0         0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1         0  is upset that he can't update his Facebook by ...
2         0  @Kenichan I dived many times for the ball. Man...
3         0    my whole body feels itchy and like its on fire 
4         0  @nationwideclass no, it's not behaving at all....


**keep only positive and negative sentiments**

In [6]:
# 0 for negative 
# 4 become 1 for positive

df = df[df.polarity != 2]
df["polarity"] = df["polarity"].map({0: 0, 4: 1})
print(df["polarity"].value_counts())

polarity
0    800000
1    800000
Name: count, dtype: int64


**Clean the Tweets**

In [10]:
#convert all the text to lowercase for consistency
def clean_text(text):
    return text.lower()
df["clean_text"] = df["text"].apply(clean_text)
print(df[["text", "clean_text"]].head())

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0  @switchfoot http://twitpic.com/2y1zl - awww, t...  
1  is upset that he can't update his facebook by ...  
2  @kenichan i dived many times for the ball. man...  
3    my whole body feels itchy and like its on fire   
4  @nationwideclass no, it's not behaving at all....  


**Train Test Split**

In [12]:
x_train, x_test, y_train, y_test = train_test_split(df["clean_text"], df["polarity"], test_size = 0.2, random_state = 42)

In [13]:
print("Train data size : ", len(x_train))
print("Test data size : ", len(x_test))

Train data size :  1280000
Test data size :  320000


**Perform Vectorization**

In [15]:
"""This code creates a TF IDF vectorizer that converts text into numerical features
using unigrams and bigrams limited to 5000 features."""

vectorizer = TfidfVectorizer(max_features = 5000, ngram_range=(1,2))

x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

print("TF-IDF shape (train):", x_train_tfidf.shape)
print("TF-IDF shape (test):", x_test_tfidf.shape)

TF-IDF shape (train): (1280000, 5000)
TF-IDF shape (test): (320000, 5000)


**Train Bernoulli Naive Bayes model**

In [16]:
bnb = BernoulliNB()
bnb.fit(x_train_tfidf, y_train)

bnb_pred = bnb.predict(x_test_tfidf)

print("bernoulli naive bayes accuracy: ", accuracy_score(y_test, bnb_pred))
print("bernoulliNB classification report: \n", classification_report(y_test, bnb_pred))

bernoulli naive bayes accuracy:  0.766478125
bernoulliNB classification report: 
               precision    recall  f1-score   support

           0       0.77      0.75      0.76    159494
           1       0.76      0.78      0.77    160506

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000



**Train Support Vector Machine (SVM) model**

In [17]:
svm = LinearSVC(max_iter = 1000)
svm.fit(x_train_tfidf, y_train)

svm_pred = svm.predict(x_test_tfidf)

print("svm accuracy: ", accuracy_score(y_test, svm_pred))
print("\n svm classification report: \n", classification_report(y_test, svm_pred))

svm accuracy:  0.79528125

 svm classification report: 
               precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



**Train Logistic Regression model**

In [18]:
logreg = LogisticRegression(max_iter = 100)
logreg.fit(x_train_tfidf, y_train)

logreg_pred = logreg.predict(x_test_tfidf)

print("logistic regression accuracy: ", accuracy_score(y_test, logreg_pred))
print("\n logistic regression classification report: \n", classification_report(y_test, logreg_pred))

logistic regression accuracy:  0.79539375

 logistic regression classification report: 
               precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



**Make Predictions on sample Tweets**

In [33]:
"""This code takes three sample tweets and transforms them into TF IDF 
features using the same vectorizer."""

sample_tweets = ["I love this!", "I hate that!", "It was okay, not great."]
sample_vec = vectorizer.transform(sample_tweets)

print("\nSample Predictions:")
print("BernoulliNB:", bnb.predict(sample_vec))
print("SVM:", svm.predict(sample_vec))
print("Logistic Regression:", logreg.predict(sample_vec))


Sample Predictions:
BernoulliNB: [1 0 1]
SVM: [1 0 1]
Logistic Regression: [1 0 1]


**1 means : positive comment, 
0 means : negitive comment**